In [4]:
import os
import google.generativeai as genai

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.0/485.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00


In [17]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [16]:
pip install --upgrade youtube-transcript-api

## Step 1a - Indexing (Document Ingestion)

In [21]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

video_id = "7xTGNNLPyMI"

try:
    # Create an instance of the API
    ytt_api = YouTubeTranscriptApi()

    # Fetch transcript using object method
    fetched = ytt_api.fetch(video_id, languages=["en"])

    # Convert to raw format (list of dicts)
    raw_transcript = fetched.to_raw_data()

    # Flatten to plain text
    transcript = " ".join(entry["text"] for entry in raw_transcript)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except NoTranscriptFound:
    print("No transcript found in the requested language.")
except VideoUnavailable:
    print("The video is unavailable.")
except Exception as e:
    print("An unexpected error occurred:", str(e))


hi everyone so I've wanted to make this video for a while it is a comprehensive but General audience introduction to large language models like Chachi PT and what I'm hoping to achieve in this video is to give you kind of mental models for thinking through what it is that this tool is it is obviously magical and amazing in some respects it's uh really good at some things not very good at other things and there's also a lot of sharp edges to be aware of so what is behind this text box you can put anything in there and press enter but uh what should we be putting there and what are these words generated back how does this work and what what are you talking to exactly so I'm hoping to get at all those topics in this video we're going to go through the entire pipeline of how this stuff is built but I'm going to keep everything uh sort of accessible to a general audience so let's take a look at first how you build something like chpt and along the way I'm going to talk about um you know som

In [22]:
raw_transcript

[{'text': "hi everyone so I've wanted to make this",
  'start': 0.719,
  'duration': 4.681},
 {'text': 'video for a while it is a comprehensive',
  'start': 2.76,
  'duration': 5.32},
 {'text': 'but General audience introduction to',
  'start': 5.4,
  'duration': 5.8},
 {'text': 'large language models like Chachi PT and',
  'start': 8.08,
  'duration': 4.519},
 {'text': "what I'm hoping to achieve in this video",
  'start': 11.2,
  'duration': 3.439},
 {'text': 'is to give you kind of mental models for',
  'start': 12.599,
  'duration': 4.641},
 {'text': 'thinking through what it is that this',
  'start': 14.639,
  'duration': 5.081},
 {'text': 'tool is it is obviously magical and',
  'start': 17.24,
  'duration': 5.199},
 {'text': "amazing in some respects it's uh really",
  'start': 19.72,
  'duration': 4.2},
 {'text': 'good at some things not very good at',
  'start': 22.439,
  'duration': 3.0},
 {'text': "other things and there's also a lot of",
  'start': 23.92,
  'duration': 4.16

## Step 1b - Indexing (Text Splitting)

In [23]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [24]:
len(chunks)

269

In [25]:
chunks[2]

Document(metadata={}, page_content="of the major llm providers like open AI anthropic and Google and so on will have some equivalent internally of something like the fine web data set so roughly what are we trying to achieve here we're trying to get ton of text from the internet from publicly available sources so we're trying to have a huge quantity of very high quality documents and we also want very large diversity of documents because we want to have a lot of knowledge inside these models so we want large diversity of high quality documents and we want many many of them and achieving this is uh quite complicated and as you can see here takes multiple stages to do well so let's take a look at what some of these stages look like in a bit for now I'd like to just like to note that for example the fine web data set which is fairly representative what you would see in a production grade application actually ends up being only about 44 terabyt of dis space um you can get a USB stick for l

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [28]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.0 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [29]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [30]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [31]:
vector_store.index_to_docstore_id

{0: '4e6358d4-ac2f-4b45-8cf9-a80d6b2022ed',
 1: '14520fa5-ec23-4ea0-8e33-c4ff0730f483',
 2: 'a67874ef-67c5-4d6d-aa11-5f2dddc0e9ae',
 3: 'c3884571-2f34-46d2-beb5-cad5ef862315',
 4: '12acccdd-7e4f-4825-ad7a-8ffcf2659401',
 5: '68969fe2-afd0-4dbf-934f-505c7703a9f4',
 6: '7dd35960-be67-4ed8-8e78-af21025de450',
 7: '9507aec8-e5d2-491a-8624-99c7e5a678ee',
 8: '91e85904-0311-46f3-83d3-f7ea87b2435c',
 9: 'ce355bdf-a45a-4d32-ad1b-a2ab31b3158d',
 10: '02865193-99b8-4fbc-80f4-42896738b281',
 11: '117e0bc1-ab06-4e42-9557-04930992e07f',
 12: '8c759736-1902-4780-a840-2193d6805f6f',
 13: '7750aa01-6a3d-405e-8016-692b6dc9ae36',
 14: 'd9d00ee6-6acb-4b06-8b75-e5b2c08689a4',
 15: '4e9a6ac3-f710-4cf6-be7e-d06cd37d5dc8',
 16: 'be6089f8-9666-45e6-8c48-45b59d4cf5cf',
 17: '2c30bcf0-4a42-46a8-91f5-b4364ed6581f',
 18: '52ac4b6f-7cce-450e-867e-12070a2992af',
 19: '9abb0a4a-80f3-46f9-a024-7b4acdb8700f',
 20: '148b1f1c-1d4d-4030-8db2-f048141f14b8',
 21: '6b2aaac1-f7c5-49df-a053-e963b96c3c6b',
 22: 'a7d6f5dc-406f-

In [32]:
vector_store.get_by_ids(['0fcb7d64-9be9-4f2c-96e1-c4aaa8310625'])

[Document(id='0fcb7d64-9be9-4f2c-96e1-c4aaa8310625', metadata={}, page_content="speaking and I think it's um it's a a question an open question as to whether the thinking strategies that are developed inside verifiable domains transfer and are generalizable to other domains that are unverifiable such as create writing the extent to which that transfer happens is unknown in the field I would say so we're not sure if we are able to do RL on everything that is very verifiable and see the benefits of that on things that are unverifiable like this prompt so that's an open question the other thing that's interesting is that this reinforcement learning here is still like way too new primordial and nent so we're just seeing like the beginnings of the hints of greatness uh in the reasoning problems we're seeing something that is in principle capable of something like the equivalent of move 37 but not in the game of Go but in open domain thinking and problem solving in principle this Paradigm is

## Step 2 - Retrieval

In [33]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x797b41d645d0>, search_kwargs={'k': 4})

In [34]:
retriever.invoke('What is deepmind')

[Document(id='6ecadf6e-f9c6-46a0-adbb-bc61d7a0d1b2', metadata={}, page_content="are deemed to be of very high quality as a source like for example Wikipedia it is very often uh the case that when you train the model you will preferentially sample from those sources so basically the model has probably done a few epochs on this data meaning that it has seen this web page like maybe probably 10 times or so and it's a bit like you like when you read some kind of a text many many times say you read something a 100 times uh then you'll be able to recite it and it's very similar for this model if it sees something way too often it's going to be able to recite it later from memory except these models can be a lot more efficient um like per presentation than human so probably it's only seen this Wikipedia entry 10 times but basically it has remembered this article exactly in its parameters okay the next thing I want to show you is something that the model has definitely not seen during its trai

## Step3 : Augmentation

In [36]:
llm = genai.GenerativeModel(model_name="gemini-1.5-flash")

prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)
retrieved_docs

[Document(id='84282369-bfd4-4951-9bcf-6c2afbe8fc8c', metadata={}, page_content="dynamical processes that have memory and so on there's no memory in this expression it's a fixed mathematical expression from input to Output with no memory it's just a stateless so these are very simple neurons in comparison to biological neurons but you can still kind of loosely think of this as like a synthetic piece of uh brain tissue if you if you like uh to think about it that way so information flows through all these neurons fire until we get to the predictions now I'm not actually going to dwell too much on the precise kind of like mathematical details of all these Transformations honestly I don't think it's that important to get into what's really important to understand is that this is a mathematical function it is uh parameterized by some fixed set of parameters like say 85,000 of them and it is a way of transforming inputs into outputs and as we twiddle the parameters we are getting uh differen

In [37]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"dynamical processes that have memory and so on there's no memory in this expression it's a fixed mathematical expression from input to Output with no memory it's just a stateless so these are very simple neurons in comparison to biological neurons but you can still kind of loosely think of this as like a synthetic piece of uh brain tissue if you if you like uh to think about it that way so information flows through all these neurons fire until we get to the predictions now I'm not actually going to dwell too much on the precise kind of like mathematical details of all these Transformations honestly I don't think it's that important to get into what's really important to understand is that this is a mathematical function it is uh parameterized by some fixed set of parameters like say 85,000 of them and it is a way of transforming inputs into outputs and as we twiddle the parameters we are getting uh different kinds of predictions and then we need to find a good setting of these\n\nare 

In [42]:
final_prompt = prompt.format(context=context_text, question=question)
final_prompt

"\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      dynamical processes that have memory and so on there's no memory in this expression it's a fixed mathematical expression from input to Output with no memory it's just a stateless so these are very simple neurons in comparison to biological neurons but you can still kind of loosely think of this as like a synthetic piece of uh brain tissue if you if you like uh to think about it that way so information flows through all these neurons fire until we get to the predictions now I'm not actually going to dwell too much on the precise kind of like mathematical details of all these Transformations honestly I don't think it's that important to get into what's really important to understand is that this is a mathematical function it is uh parameterized by some fixed set of parameters like say 85,000 of them and it is a way of tran

## Step 4: Generation

In [43]:
answer = llm.generate_content(final_prompt)
print(answer.text)

I don't know.  The provided text discusses neural networks and how they process and "remember" information, but makes no mention of nuclear fusion.



### Building a Chain

In [44]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [45]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [50]:
def call_gemini(prompt_text):
    response = llm.generate_content(prompt_text)
    return response.text

gemini_runnable = RunnableLambda(call_gemini)

parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [52]:
from langchain_core.prompts import PromptTemplate

# Convert PromptTemplate to string using .format()
format_prompt = RunnableLambda(lambda x: prompt.format(**x))

# Now build the chain properly
main_chain = parallel_chain | format_prompt | gemini_runnable | StrOutputParser()

In [55]:
response = main_chain.invoke("Can you please summarize the video?")
print(response)

The video discusses how large language models generate text.  The models are stochastic, meaning they sample and "flip coins" at each step, creating outputs that are statistically similar to the training data but not identical.  Frequently occurring information from high-quality sources like Wikipedia is more likely to be reproduced accurately because the model has "seen" it many times during training.  However, the model's knowledge is vague and probabilistic; it's a recollection of internet documents, not precise storage.  The video also mentions the importance of practice problems in learning, where the goal is to discover the solution process, not just the final answer.

